In [2]:
import pandas as pd

In [8]:
# Get SNP symbols
dls = "http://www.cboe.com/products/weeklys-options/available-weeklys"
data = pd.read_html(dls)


,symbol
0,UVIX
1,MMM
2,ABT
3,ABBV
4,ANF
...,...
625,ZM
626,ZS
627,VXX
628,INDA


In [11]:
df = data[0].rename(columns={"STOCK SYMBOL": "symbol"})[["symbol"]]
df_weeklies = pd.concat(
    [
        df.assign(secType="STK", exchange="SMART"),
        df.assign(secType="IND", exchange="CBOE"),
    ],
    ignore_index=True,
)

df_weeklies = df_weeklies.assign(
    symbol=df_weeklies.symbol.str.replace("[^a-zA-Z]", "", regex=True))

# Generate the snp 500s
try:
    s500 = list(
        pd.read_html(
            "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            header=0,
            match="Symbol",
        )[0].loc[:, "Symbol"])
except Exception as e:
    print(f"Error: {e}")

# without dot in symbol
snp500 = [s.replace(".", "") if "." in s else s for s in s500]

# Keep only equity weeklies that are in S&P 500, and all indexes in the weeklies
df_symlots = df_weeklies[((df_weeklies.secType == "STK") &
                            (df_weeklies.symbol.isin(snp500)))
                            | (df_weeklies.secType == "IND")].reset_index(
                                drop=True)

In [12]:
df_symlots

,symbol,secType,exchange
0,MMM,STK,SMART
1,ABT,STK,SMART
2,ABBV,STK,SMART
3,ACN,STK,SMART
4,ATVI,STK,SMART
...,...,...,...
856,ZM,IND,CBOE
857,ZS,IND,CBOE
858,VXX,IND,CBOE
859,INDA,IND,CBOE
